# Feature Engineering

### Imports

In [2]:
import pandas as pd
import geopandas as gpd
import zipfile
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.config import RAIN_COLS_HEB_TO_ENG
from src.feature_engineering import modify_rainfall_df_values, group_rainfall_by_day_and_hour, merge_rain_to_trips_df
from src.feature_engineering import route_to_shape_dict, get_linestring_for_shape, add_linestrings_to_route_dict,extract_linestring
from src.feature_engineering import buffer_and_dissolve_routes, calc_length_within_buffer, calc_curvity, parse_mixed_date, filter_by_multiple_date_windows

In [17]:
### Import Trips interim dataframe
df_trips = pd.read_csv(r'../data/interim_data/telaviv_buses_april_2024_cleaned.csv')

### Rainfall
Rainfall data was obtained from the Israeli Meteorological Service (IMS) open data portal:
https://ims.gov.il/he/data_gov

In [18]:
### Importing rainfall data collected by the Israeli Meteorological Service (IMS)
df_rain = pd.read_csv(r'../data/external/rainfall_tlv_apr_2024.csv')

### Renaming its columns to english for easy processing
df_rain = df_rain.rename(columns=RAIN_COLS_HEB_TO_ENG)

### Modifying value types
df_rain = modify_rainfall_df_values(df_rain)

### Grouping (sum) rainfall by day and hour 
rain_grouped_df = group_rainfall_by_day_and_hour(df_rain)

### Merge the rainfall (mm) to th trips df - so each trip will have the amount of rainfall at the time (hour) of the trip
df_trips = merge_rain_to_trips_df(rain_grouped_df,df_trips)

## Spatial attributes

### Imports

In [20]:
#### public transport paths
public_transport_paths = gpd.read_file("../data/external/shapefiles/Nataz_shape_202601.zip")


#### gtfs data
zip_path = '../data/external/gtfs/gtfs.zip'
with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())  # see file names inside

    df_trips_gtfs = pd.read_csv(z.open('trips.txt'))
    df_shapes_gtfs = pd.read_csv(z.open('shapes.txt'))
    df_routes_gtfs = pd.read_csv(z.open('routes.txt'))

['routes.txt', 'shapes.txt', 'trips.txt']


### Process

- Assign to each trips its geometry
- Convert  the trips df to GDF (GeoDataFrame) based on the route geometry (linestring)

In [21]:
### Create a dictionary to each route_id the associated shape_id in gtfs {'route_id_1': {'shape_id': shape_id_1},...}                                                                                    
route_shape_map  = route_to_shape_dict(df_trips_gtfs, df_trips.route_id.unique())

### Add to the route_shape_map to each route_id the linestinrg extracted from shape_id df {'shape_id': shape_id_1, 'linestring': 'linestring_1},...} 
route_shape_line_map = add_linestrings_to_route_dict(df_shapes_gtfs, route_shape_map)

### Add this linestring to the trips df as an attribute - "linestring"
df_trips["linestring"] = df_trips["route_id"].map(lambda x: extract_linestring(x, route_shape_line_map))

### Convert the df to GeoDataFrame, using the linestring as the geometry reference
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="linestring", crs="EPSG:4326")

### Handle CRS - make it in ITM for better distance measuring
gdf_trips = gdf_trips.set_crs(epsg=4326)   # only if not already set
gdf_trips = gdf_trips.to_crs(epsg=2039)

- Handle The Public Transport Paths data
- Filter paths that were removed before, or added after the investigated timeframe

In [23]:
public_transport_routes = public_transport_paths.set_crs(epsg=2039, allow_override=True)

cols = ["STARTDATE", "ENDDATE"]
public_transport_paths[cols] = public_transport_paths[cols].apply(lambda col: col.apply(parse_mixed_date))

public_transport_routes_filtered= filter_by_multiple_date_windows(
                                                                public_transport_paths,
                                                                start_cols=["STARTDATE"],
                                                                end_cols=["ENDDATE"],
                                                                start_cutoffs=["2024-01-01"],
                                                                end_cutoffs=["2024-02-01"]
)

- Adding spatial attributes
- Calculating the length within public transport path (based on buffer intersection with it)
- Adding the percentage of the route within this path
- Adding curvity metric (route_length/aerial distance between start and end stops)

In [24]:
### Making 15 m buffer around each public transport path
pt_routes_area_gdf = buffer_and_dissolve_routes(public_transport_routes_filtered, buffer_m=15)

### Dissolving all the geometry into one polygon
buffer_geom = pt_routes_area_gdf.geometry.iloc[0]

### Intersection each route linestring, and calculating the sum og the linestring inside the buffer
### lines lower than 200m removed
gdf_trips["length_in_buffer_m"] = gdf_trips["linestring"].apply(
    lambda geom: calc_length_within_buffer(geom, buffer_geom, min_length=200)
)

### Adding the gtfs route_length attribut in meters
gdf_trips['route_length'] = gdf_trips.length

### Calculating the percetnage of the distance inside the public transport path from the entire route length
gdf_trips['perc_within_pt_route'] = round(gdf_trips['length_in_buffer_m']/gdf_trips['route_length']*100,2)

### Curvity metric
gdf_trips["curvity"] = gdf_trips["linestring"].apply(calc_curvity)

## Other attributes

- Adding one attribute that will consist -agency + line number + direction-
- Urban - boolean  (more than 25km)

In [26]:
gdf_trips['line_num_agency_alter_dir'] = gdf_trips['agency_name'] +'_'+ gdf_trips['line_num'] +'_'+ gdf_trips['direction'].astype(str)
gdf_trips['urban'] = gdf_trips['route_length'] <=25

In [27]:
gdf_trips.drop(columns = ["linestring"]).to_csv(r'../data/interim_data/telaviv_buses_0704_1304_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')

In [28]:

gdf_trips[['route_id','linestring']].drop_duplicates(subset=['route_id','linestring']).to_csv(r'../data/interim_data/routes_linestrings.csv',index=False,encoding='utf-8-sig')

In [26]:
df_trips = pd.read_csv(r'../data/interim_data/telaviv_buses_0704_1304_2024_cleaned_new_features.csv')

In [27]:
df_trips.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,gtfs_route_id,gtfs_ride_id,SIRI_id,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban
0,2024-04-07,Sunday,0,826,ת.מרכזית תל אביב קומה 7/רציפים-תל אביב יפו<->ד...,15123,10826,1,9,Egged,...,4666540,72316307,67073403,0.0,30946.992877,128214.134588,24.14,1.409145,Egged_826_1,False
1,2024-04-07,Sunday,0,61,מסוף עמידר-רמת גן<->מסוף כרמלית/הורדה-תל אביב ...,2512,16061,2,0,Dan,...,4664121,72282601,67073416,0.0,2595.473725,12742.652985,20.37,1.841781,Dan_61_2,False
2,2024-04-07,Sunday,0,63,הכובשים/דניאל-תל אביב יפו<->מסוף אלוף שדה-רמת ...,2517,17063,1,0,Dan,...,4657330,72178343,67073417,0.0,2833.152306,12232.226796,23.16,1.842028,Dan_63_1,False
3,2024-04-07,Sunday,0,70,ת.רכבת תל אביב - סבידור/רציפים C-תל אביב יפו<-...,2542,20070,1,0,Dan,...,4657339,72179065,67073420,0.0,4872.654308,10968.248411,44.43,1.791541,Dan_70_1,False
4,2024-04-07,Sunday,0,66,ת. מרכזית פ''ת/רציפים-פתח תקווה<->מסוף כרמלית/...,2532,15066,1,0,Dan,...,4664127,72276232,67073424,0.0,7989.261100,16464.450441,48.52,1.394338,Dan_66_1,False


In [28]:
df_trips['route_dir_alt_day_hr'] = (
    df_trips['route_mkt'].fillna(0).astype(int).astype(str) + '_' +
    df_trips['direction'].fillna(0).astype(int).astype(str) + '_' +
    df_trips['alternative'].astype(str) + '_' +
    df_trips['day'].astype(str) + '_' +
    df_trips['hour_rounded'].fillna(0).astype(int).astype(str)
)

print(df_trips['route_dir_alt_day_hr'].head())

0    10826_1_9_Sunday_0
1    16061_2_0_Sunday_0
2    17063_1_0_Sunday_0
3    20070_1_0_Sunday_0
4    15066_1_0_Sunday_0
Name: route_dir_alt_day_hr, dtype: object


In [29]:
validation = pd.read_csv(r'../data/raw_data/entire_data/grouped_validations.csv')

C:\Users\Tamir\AppData\Local\Temp\ipykernel_45504\2619286534.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  validation = pd.read_csv(r'../data/raw_data/entire_data/grouped_validations.csv')


In [30]:
validation.head()

,Alternative,RouteId,Direction,FullHour,DayName,Total_Passengers,Bus_Count,Avg_Passengers_Per_Bus
0,#,10001,3,3.0,Friday,42,2,21.000000
1,#,10001,3,3.0,Monday,18,3,6.000000
2,#,10001,3,3.0,Sunday,41,3,13.666667
3,#,10001,3,3.0,Thursday,31,3,10.333333
4,#,10001,3,3.0,Tuesday,47,3,15.666667


In [32]:
validation['route_dir_alt_day_hr'] = (
    validation['RouteId'].fillna(0).astype(int).astype(str) + '_' +
    validation['Direction'].fillna(0).astype(int).astype(str) + '_' +
    validation['Alternative'].astype(str) + '_' +
    validation['DayName'].astype(str) + '_' +
    validation['FullHour'].fillna(0).astype(int).astype(str)
)

print(validation['route_dir_alt_day_hr'].head())

0      10001_3_#_Friday_3
1      10001_3_#_Monday_3
2      10001_3_#_Sunday_3
3    10001_3_#_Thursday_3
4     10001_3_#_Tuesday_3
Name: route_dir_alt_day_hr, dtype: object


In [33]:
df_trips = pd.merge(
    df_trips,
    validation[['route_dir_alt_day_hr', 'Total_Passengers', 'Avg_Passengers_Per_Bus']],
    on='route_dir_alt_day_hr',
    how='left'
)

print(f"גודל הטבלה אחרי merge: {df_trips.shape}")
print(df_trips.head())

גודל הטבלה אחרי merge: (92548, 37)
         date     day  hour_rounded line_num  \
0  2024-04-07  Sunday             0      826   
1  2024-04-07  Sunday             0       61   
2  2024-04-07  Sunday             0       63   
3  2024-04-07  Sunday             0       70   
4  2024-04-07  Sunday             0       66   

                                           line_name  route_id  route_mkt  \
0  ת.מרכזית תל אביב קומה 7/רציפים-תל אביב יפו<->ד...     15123      10826   
1  מסוף עמידר-רמת גן<->מסוף כרמלית/הורדה-תל אביב ...      2512      16061   
2  הכובשים/דניאל-תל אביב יפו<->מסוף אלוף שדה-רמת ...      2517      17063   
3  ת.רכבת תל אביב - סבידור/רציפים C-תל אביב יפו<-...      2542      20070   
4  ת. מרכזית פ''ת/רציפים-פתח תקווה<->מסוף כרמלית/...      2532      15066   

   direction alternative agency_name  ... rainfall_mm length_in_buffer_m  \
0          1           9       Egged  ...         0.0       30946.992877   
1          2           0         Dan  ...         0.0        

In [34]:
df_trips.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban,route_dir_alt_day_hr,Total_Passengers,Avg_Passengers_Per_Bus
0,2024-04-07,Sunday,0,826,ת.מרכזית תל אביב קומה 7/רציפים-תל אביב יפו<->ד...,15123,10826,1,9,Egged,...,0.0,30946.992877,128214.134588,24.14,1.409145,Egged_826_1,False,10826_1_9_Sunday_0,1.0,1.0
1,2024-04-07,Sunday,0,61,מסוף עמידר-רמת גן<->מסוף כרמלית/הורדה-תל אביב ...,2512,16061,2,0,Dan,...,0.0,2595.473725,12742.652985,20.37,1.841781,Dan_61_2,False,16061_2_0_Sunday_0,13.0,6.5
2,2024-04-07,Sunday,0,63,הכובשים/דניאל-תל אביב יפו<->מסוף אלוף שדה-רמת ...,2517,17063,1,0,Dan,...,0.0,2833.152306,12232.226796,23.16,1.842028,Dan_63_1,False,17063_1_0_Sunday_0,9.0,9.0
3,2024-04-07,Sunday,0,70,ת.רכבת תל אביב - סבידור/רציפים C-תל אביב יפו<-...,2542,20070,1,0,Dan,...,0.0,4872.654308,10968.248411,44.43,1.791541,Dan_70_1,False,20070_1_0_Sunday_0,5.0,5.0
4,2024-04-07,Sunday,0,66,ת. מרכזית פ''ת/רציפים-פתח תקווה<->מסוף כרמלית/...,2532,15066,1,0,Dan,...,0.0,7989.261100,16464.450441,48.52,1.394338,Dan_66_1,False,15066_1_0_Sunday_0,12.0,12.0


In [36]:
df_trips.to_csv(r'../data/interim_data/telaviv_buses_0704_1304_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')